# Toto 2.0 on GIFT-Eval

## Setup

1. Install GIFT-Eval as instructed in the [README](../README.md).

2. Download the GIFT-Eval dataset:
```bash
huggingface-cli download Salesforce/GiftEval --repo-type=dataset --local-dir /path/to/gifteval
```

3. Install Toto 2.0:
```bash
pip install "toto-2 @ git+https://github.com/DataDog/toto.git#subdirectory=toto2"
```

4. Set the `GIFT_EVAL` environment variable to the dataset path (see cell below).

In [ ]:
import os
import json
import gc
import math

import numpy as np
import pandas as pd
import torch
from gluonts.ev.metrics import (
    MAE, MAPE, MASE, MSE, MSIS, ND, NRMSE, RMSE, SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality

from gift_eval.data import Dataset
from toto2 import Toto2Model, Toto2GluonTSModel, Toto2GluonTSModelConfig

os.environ["GIFT_EVAL"] = "Change/To/GiftEval/Local/Path"

In [ ]:
SIZE = "22m"  # 4m | 22m | 313m | 1B | 2.5B | 2.5B-FT
MODEL_NAME = f"Toto-2.0-{SIZE}"
CHECKPOINT = f"Datadog/{MODEL_NAME}"
DEFAULT_CONTEXT_LENGTH = 4096
BATCH_SIZE = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PRETTY_DATASET_NAMES = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

SHORT_DATASETS = "m4_weekly"
MED_LONG_DATASETS = "bizitobs_l2c/H"

# SHORT_DATASETS = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
# MED_LONG_DATASETS = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"

In [ ]:
METRICS = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(forecast_type="mean"),
    NRMSE(forecast_type="mean"),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

METRIC_COLUMNS = [
    "MSE[mean]", "MSE[0.5]", "MAE[0.5]", "MASE[0.5]", "MAPE[0.5]",
    "sMAPE[0.5]", "MSIS", "RMSE[mean]", "NRMSE[mean]", "ND[0.5]",
    "mean_weighted_sum_quantile_loss",
]

In [ ]:
def evaluate_dataset(model, dataset_name: str, term: str, properties: dict) -> dict | None:
    ds = Dataset(name=dataset_name, term=term, to_univariate=False, storage_env_var="GIFT_EVAL")

    gts_config = Toto2GluonTSModelConfig(
        prediction_length=ds.prediction_length,
        context_length=DEFAULT_CONTEXT_LENGTH,
        target_dim=ds.target_dim,
        past_feat_dynamic_real_dim=ds.past_feat_dynamic_real_dim,
    )
    gts_model = Toto2GluonTSModel(model, gts_config).to(DEVICE).eval()
    predictor = gts_model.create_predictor(batch_size=BATCH_SIZE, device=DEVICE)

    # parse dataset key
    if "/" in dataset_name:
        ds_key = PRETTY_DATASET_NAMES.get(dataset_name.split("/")[0].lower(), dataset_name.split("/")[0].lower())
        ds_freq = dataset_name.split("/")[1]
    else:
        ds_key = PRETTY_DATASET_NAMES.get(dataset_name.lower(), dataset_name.lower())
        ds_freq = properties[ds_key]["frequency"]

    ds_config = f"{ds_key}/{ds_freq}/{term}"
    print(f"Evaluating {ds_config} (pred={ds.prediction_length}, dim={ds.target_dim})")

    try:
        res = evaluate_model(
            predictor,
            test_data=ds.test_data,
            metrics=METRICS,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=get_seasonality(ds.freq),
        )
    except Exception as e:
        print(f"  FAILED: {e}")
        return None

    row = {"dataset": ds_config, "model": MODEL_NAME}
    for col in METRIC_COLUMNS:
        row[f"eval_metrics/{col}"] = res[col].iloc[0]
    row["domain"] = properties[ds_key]["domain"]
    row["num_variates"] = properties[ds_key]["num_variates"]
    print(f"  MASE={row['eval_metrics/MASE[0.5]']:.4f}")
    return row

In [ ]:
properties = json.load(open("dataset_properties.json"))

model = Toto2Model.from_pretrained(CHECKPOINT, map_location=DEVICE)
model = model.to(DEVICE).eval()

short_datasets = SHORT_DATASETS.split()
med_long_datasets = MED_LONG_DATASETS.split()
all_datasets = list(set(short_datasets + med_long_datasets))

rows = []
for dataset_name in all_datasets:
    for term in ["short", "medium", "long"]:
        if term in ("medium", "long") and dataset_name not in med_long_datasets:
            continue
        row = evaluate_dataset(model, dataset_name, term, properties)
        if row is not None:
            rows.append(row)

del model
torch.cuda.empty_cache()
gc.collect()

results = pd.DataFrame(rows)
results

In [ ]:
os.makedirs("../../results/gift_eval/toto_2_0", exist_ok=True)
results.to_csv("../../results/gift_eval/toto_2_0/all_results.csv", index=False)
print("Saved results.")